In [5]:
import sys
import os
import MetaTrader5 as mt5
import time

project_dir = os.path.abspath("..")
if project_dir not in sys.path:
    sys.path.append(project_dir)

import polars as pl

from src.infra.mtBase import mtBase

In [6]:
mtb = mtBase(
    account="darwinexzero_acc",
    credentials_path=os.path.join("..", "secrets", "mt5_acc_cred.yaml"),
    config_path=os.path.join("..", "secrets", "mt5_config.ini"),
)
mtb.mt5_init()

MetaTrader 5 connection established


In [3]:
info_msft = mtb.get_symbol_info("MSFT")
info_clf = mtb.get_symbol_info("CLF")

print(info_clf["trade_mode"])
print(info_msft["trade_mode"])

3
4


In [4]:
info_msft

{'symbol': 'MSFT',
 'visible': True,
 'trade_mode': 4,
 'digits': 2,
 'point': 0.01,
 'time': 0,
 'volume': 0,
 'volume_real': 0.0,
 'trade_contract_size': 1.0,
 'volume_min': 1.0,
 'volume_max': 1000.0,
 'volume_step': 1.0,
 'currency_base': 'USD',
 'currency_profit': 'USD',
 'currency_margin': 'USD',
 'trade_stops_level': 0,
 'trade_freeze_level': 0,
 'filling_mode': 3,
 'expiration_mode': 15,
 'category': ''}

In [7]:
mtb.get_account_info()._asdict()

{'login': 4000075958,
 'trade_mode': 2,
 'leverage': 1,
 'limit_orders': 400,
 'margin_so_mode': 0,
 'trade_allowed': True,
 'trade_expert': True,
 'margin_mode': 2,
 'currency_digits': 2,
 'fifo_close': False,
 'balance': 1003710.53,
 'credit': 0.0,
 'profit': 15810.9,
 'equity': 1019521.43,
 'margin': 783124.04,
 'margin_free': 220586.49,
 'margin_level': 130.18645551986884,
 'margin_so_call': 0.0,
 'margin_so_so': 0.0,
 'margin_initial': 0.0,
 'margin_maintenance': 0.0,
 'assets': 0.0,
 'liabilities': 0.0,
 'commission_blocked': 0.0,
 'name': 'kimeri_MT5',
 'server': 'Darwinex-Live',
 'currency': 'USD',
 'company': 'Tradeslide Trading Tech Limited'}

In [ ]:
mtb.get_symbol_price("MSFT")

In [4]:
symbols = mt5.symbols_get()
symbols_dict = [s._asdict() for s in symbols]
symbols_name = [s.name for s in symbols if s.trade_mode == mt5.SYMBOL_TRADE_MODE_FULL]

In [5]:
symbols_stocks = [s for s in symbols_dict if "Stock" in s['path'] and s['trade_mode'] == mt5.SYMBOL_TRADE_MODE_FULL]

print(f"Total symbols: {len(symbols)}")
print(f"Total stock symbols: {len(symbols_stocks)}")

stocks = [s['name'] for s in symbols_stocks]
for s in stocks:
    print(s)

Total symbols: 811
Total stock symbols: 687
ACHC
ADBE
ADI
ADP
ADSK
AKAM
ALGN
ALNY
AMAT
AMD
AMGN
AMKR
AMZN
APA
APLS
ARCC
ARWR
AVGO
AXON
AZTA
BIIB
BKNG
BL
BLDR
BLK
BMRN
BRKR
CACC
CAR
CASY
CDNS
CDW
CG
CGNX
CHDN
CHRW
CHTR
CINF
CMCSA
CME
COLM
COST
CPRT
CROX
CRWD
CSGP
CSX
CTAS
CTSH
DBX
DDOG
DLTR
DNLI
DOCU
DOX
DXCM
EA
EBAY
EEFT
ENPH
ENTG
ETSY
EWBC
EXEL
A
AA
AAP
ABBV
ABT
ACM
ACN
ADM
AEP
AES
AFG
AFL
AGCO
AIG
AIZ
AJG
AL
ALB
ALK
ALL
ALLY
AME
AMG
AMP
AMT
AN
ANET
AON
AOS
APD
APH
ARES
ARMK
ARW
ASH
AVTR
AVY
AWI
AWK
AXTA
AYI
AZO
BAC
BAH
BALL
BAX
BBY
BC
BDX
BEN
BFAM
BILL
BIO
BJ
BK
BKR
BLD
BMY
BR
BRKb
BRO
BSX
BURL
BWA
BX
BYD
C
CABO
CACI
CAG
CAH
CARR
CB
CBRE
CC
CCI
CCK
CCL
CE
CF
CFG
CFR
CHD
CHE
CHWY
CI
CIEN
CL
CLX
CMA
CMG
CMI
CMS
CNC
CNP
COF
COHR
COO
COP
COR
CPAY
CRL
CRM
CSL
CTLT
CTVA
CVNA
CVS
D
DAL
DAR
DAY
DD
DE
DECK
DELL
DG
DGX
DHI
DHR
DKS
DLB
DOV
DPZ
DRI
DT
DTE
DUK
DVA
DVN
DXC
ECL
EFX
EHC
EIX
EL
ELV
EME
EMN
EMR
ENOV
EOG
EPAM
EQH
EQT
ES
ESI
ESNT
ESTC
ETN
ETR
EVR
EVRG
EW
EXC
EXP
EXPD
EXPE
FANG
FAST
FFIV

In [ ]:
spreads = {}
for stock in stocks:
    n_retries = 5
    for i in range(n_retries):
        mt5.symbol_select(stock, True)
        time.sleep(0.2)
        tick = mt5.symbol_info_tick(stock)._asdict()
        mt5.symbol_select(stock, False)
        time.sleep(0.2)  # to avoid overloading MT5 API
        if tick is None:
            _, le = mt5.last_error()
            print(f"Tick data not available for symbol: {stock}, error: {le}")
            continue

        # check that tick has 'ask' and 'bid' keys
        if 'ask' not in tick or 'bid' not in tick:
            print(f"Tick data incomplete for symbol: {stock}")
            continue

        ask = tick.get('ask', None)
        bid = tick.get('bid', None)
        if ask is None or bid is None:
            print(f"Tick data incomplete for symbol: {stock}")
            continue

        rel_spread = (ask - bid) / (bid + 1e-8)
        if rel_spread <= 1e-6:
            _, le = mt5.last_error()
            print(f"Unrealistic spread for symbol: {stock}, ask: {ask}, bid: {bid}, spread: {rel_spread}, error: {le}")
            continue
        else:
            break
    print(f"Symbol: {stock}, Ask: {ask}, Bid: {bid}, Spread: {rel_spread}")

    spreads[stock] = rel_spread


Symbol: ACHC, Ask: 23.47, Bid: 23.44, Spread: 0.001279863480682549
Symbol: ADBE, Ask: 354.14, Bid: 353.95, Spread: 0.0005367989828920179
Symbol: ADI, Ask: 240.3, Bid: 240.12, Spread: 0.0007496251873751065
Symbol: ADP, Ask: 283.89, Bid: 283.76, Spread: 0.0004581336340407884
Symbol: ADSK, Ask: 308.18, Bid: 307.96, Spread: 0.0007143784906899712
Symbol: AKAM, Ask: 74.84, Bid: 74.7, Spread: 0.001874163319695568
Symbol: ALGN, Ask: 135.51, Bid: 135.29, Spread: 0.0016261364475108101
Symbol: ALNY, Ask: 466.33, Bid: 463.99, Spread: 0.005043212138083887
Symbol: AMAT, Ask: 220.46, Bid: 220.32, Spread: 0.0006354393609007825
Symbol: AMD, Ask: 229.89, Bid: 229.84, Spread: 0.00021754263834757883
Symbol: AMGN, Ask: 295.92, Bid: 295.61, Spread: 0.0010486790027046293
Symbol: AMKR, Ask: 30.76, Bid: 30.75, Spread: 0.000325203251926814
Symbol: AMZN, Ask: 217.94, Bid: 217.9, Spread: 0.00018357044514986844
Symbol: APA, Ask: 22.88, Bid: 22.87, Spread: 0.0004372540444086344
Symbol: APLS, Ask: 28.13, Bid: 28.12,

In [7]:
# save dict as csv
df_spreads = pl.DataFrame({
    "symbol": list(spreads.keys()),
    "rel_spread": list(spreads.values())
})
df_spreads.write_csv("stock_spreads.csv")

In [8]:
spreads_arr = df_spreads['rel_spread'].to_numpy()
print(f"Average stock spread: {spreads_arr.mean()}")
for q in [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
    print(f"{q} quantile stock spread: {pl.Series(spreads_arr).quantile(q)}")

Average stock spread: 0.0022164058558563336
0.05 quantile stock spread: 0.00019602718242316299
0.1 quantile stock spread: 0.0002491280518062785
0.25 quantile stock spread: 0.00047092064984829345
0.5 quantile stock spread: 0.0015920186795169338
0.75 quantile stock spread: 0.003296820163660671
0.9 quantile stock spread: 0.005187991838811462
0.95 quantile stock spread: 0.00665444692354735


In [9]:
mask_50 = spreads_arr < 0.0015
filtered_df = df_spreads.filter(mask_50)
filtered_df.write_csv("stock_spreads_below_median.csv")

In [10]:
filtered_sym = filtered_df['symbol']
for s in filtered_sym:
    print(s)

ACHC
ADBE
ADI
ADP
ADSK
AMAT
AMD
AMGN
AMKR
AMZN
APA
APLS
ARCC
AVGO
BIIB
BLDR
BMRN
CHRW
CMCSA
COLM
COST
CPRT
CROX
CSGP
CSX
CTAS
CTSH
DBX
DNLI
DOCU
DXCM
EA
EBAY
ENPH
ETSY
A
AA
AAP
ABBV
ABT
ACN
ADM
AEP
AES
AFL
AIG
AJG
AL
ALB
ALK
AME
AMP
AMT
ANET
AON
APD
APH
AVTR
AWK
AXTA
BAC
BAX
BBY
BC
BDX
BEN
BILL
BIO
BK
BKR
BR
BRKb
BRO
BSX
BX
BYD
C
CAG
CAH
CARR
CB
CBRE
CCI
CCL
CF
CFG
CHD
CHWY
CI
CL
CLX
CMG
CMS
CNC
CNP
COF
COO
COP
COR
CTLT
CTVA
CVS
D
DAL
DAR
DAY
DD
DE
DECK
DG
DHI
DT
DUK
DVN
DXC
ECL
EHC
EME
EMN
EMR
ENOV
EOG
EQH
ES
ESTC
ETN
EVRG
EW
EXC
EXPD
FANG
FAST
F
FBIN
FCX
FE
FITB
FIVE
FIVN
FLEX
FMC
FOXA
FRPT
FTNT
GAP
GD
GEN
GH
GIS
GLW
GM
GILD
GNTX
GPK
GOOG
GOOGL
GTLS
HBAN
HAL
HELE
HOG
HOLX
HPE
HPQ
HUBS
HUM
HUN
ICE
INCY
IP
IPG
JKHY
KHC
KLAC
LKQ
LNT
IT
IVZ
J
JBL
JCI
K
KDP
KEY
KKR
KMI
KMX
KNX
KR
LII
LLY
LMT
LOW
LVS
LYFT
M
MA
MAN
MAT
MCK
MCHP
MDLZ
MDT
MET
MGM
MHK
META
MNST
MMC
MO
MOS
MRNA
MRVL
MS
MSTR
MTZ
MU
NBIX
NCLH
NEE
NI
NFLX
NTNX
NSC
NVDA
NWSA
NYT
OC
ODFL
OKE
OLN
OMCL
OMF
ON
ORCL
ORI
ORLY
OXY
PANW
PA